# 12: Verifying AI-Generated Code for Scientific Computing

In this notebook we learn systematic methods for verifying code produced by AI coding assistants. When an AI writes a function that computes drawdown, estimates recharge, or processes streamflow data, how do you know the answer is correct?

**Learning objectives:**
- Understand *why* verification is essential when using AI for hydrology code
- Apply three verification strategies: analytical solutions, unit testing, and dimensional analysis
- Write pytest test functions that catch common AI errors in hydrology equations
- Use inline assertions as a rapid feedback loop when iterating with AI

**Time estimate:** ~20 minutes

> **Note:** All AI-generated code examples in this notebook were captured ahead of time. No live AI API calls are made. This notebook works without any AI tool installed.

## 1. Why Verify?

AI coding assistants produce code that *looks* correct. It runs without syntax errors, uses the right library imports, and even includes docstrings. But "runs without errors" is a dangerously low bar for scientific computing.

### The hydrology-specific problem

In hydrology, a wrong answer can look perfectly reasonable:

| Scenario | AI output | Correct answer | Consequence if undetected |
|----------|-----------|---------------|--------------------------|
| Well drawdown at 1 km | 14.06 m | 1.41 m | Overestimate pumping impact by 10x |
| Streamflow unit conversion | 28.32 cms | 2.832 cms | Wrong flood frequency analysis |
| Darcy flux through aquitard | -0.003 m/day | 0.003 m/day | Flow direction reversed in model |

These errors share a pattern: the code runs, the output is a number with the right units, and unless you *check* against a known value, you would never notice.

### Three reasons verification matters more with AI

1. **AI is confidently wrong.** Unlike a compiler error or a traceback, AI errors are silent. The code executes and returns a plausible-looking number.

2. **AI doesn't understand physics.** It cannot tell you that negative hydraulic conductivity is impossible, or that drawdown should increase (not decrease) with pumping rate.

3. **AI errors are subtle.** The most dangerous AI bugs are not `NameError` or `TypeError` — they are using $K$ where $T$ is needed, or applying the wrong unit conversion factor. These require domain knowledge to catch.

## 2. Strategy 1: Compare Against Known Answers for Analytical Solutions

If an AI writes code to compute drawdown, and you know the Theis solution gives $s = 1.40636669$ m for specific parameters, you can check the AI output against that value immediately.

### Example: Verifying AI-generated Theis equation

In [ ]:
import numpy as np
from scipy.special import exp1

# AI-generated function (pre-captured)
def theis_drawdown(Q, T, S, r, t):
    """Compute drawdown using the Theis equation.
    
    Parameters
    ----------
    Q : float
        Pumping rate in m³/day.
    T : float
        Transmissivity in m²/day.
    S : float
        Storativity (dimensionless).
    r : float
        Distance from pumping well in meters.
    t : float
        Time since pumping began in days.
    
    Returns
    -------
    float
        Drawdown in meters.
    """
    u = (r**2 * S) / (4.0 * T * t)
    s = (Q / (4.0 * np.pi * T)) * exp1(u)
    return s

In [ ]:
# Verify against the known analytical solution
s = theis_drawdown(Q=4088, T=1000, S=3e-4, r=1000, t=10)
expected = 1.40636669

print(f"AI function result:    {s:.8f} m")
print(f"Analytical solution:   {expected:.8f} m")
print(f"Match (within 1e-6):   {np.isclose(s, expected, rtol=1e-6)}")

### Example: Verifying AI-generated Darcy's Law

Darcy's Law describes the volumetric flow rate through a porous medium:

$$Q = -K \cdot A \cdot \frac{dh}{dL}$$

where $Q$ is the volumetric flow rate [L³/T], $K$ is hydraulic conductivity [L/T], $A$ is the cross-sectional area [L²], and $dh/dL$ is the hydraulic gradient [L/L].

In [ ]:
# AI-generated function (pre-captured)
def darcy_flow(K, A, dh, dL):
    """Compute volumetric flow rate using Darcy's Law.
    
    Parameters
    ----------
    K : float
        Hydraulic conductivity in m/day.
    A : float
        Cross-sectional area in m².
    dh : float
        Head difference in meters (h_upstream - h_downstream).
    dL : float
        Flow path length in meters.
    
    Returns
    -------
    float
        Volumetric flow rate in m³/day (positive = flow from high to low head).
    """
    gradient = dh / dL
    Q = K * A * gradient  # Note: positive Q when dh > 0 (flow downhill)
    return Q

In [ ]:
# Verify against a hand calculation
# K = 10 m/day, A = 50 m², dh = 5 m over dL = 100 m
# Q = 10 * 50 * (5/100) = 25 m³/day
Q = darcy_flow(K=10, A=50, dh=5, dL=100)
expected_Q = 25.0

print(f"AI function result:    {Q:.2f} m³/day")
print(f"Hand calculation:      {expected_Q:.2f} m³/day")
print(f"Match: {np.isclose(Q, expected_Q)}")

**Key takeaway:** Whenever you ask AI to implement a well-known equation, compute the answer by hand (or look it up) for at least one set of parameters. If the AI output does not match, you have a bug to find.

## 3. Strategy 2: Unit Testing with pytest

Analytical spot-checks are great for a quick sanity check, but they do not scale. What if you need to verify the function works across a range of inputs? What if you refactor the code later?

**Unit tests** are small, automated functions that check specific behaviors. Python's `pytest` framework makes them easy to write.

### Writing pytest functions for hydrology equations

A pytest test function:
- Starts with `test_`
- Uses `assert` statements to check expected behavior
- Can be run automatically any time the code changes

Here is how you would test the Theis equation implementation:

In [ ]:
# Save this to a file called test_theis.py, or run with %%ipytest

def test_theis_known_value():
    """Test Theis drawdown against the known analytical solution."""
    s = theis_drawdown(Q=4088, T=1000, S=3e-4, r=1000, t=10)
    assert np.isclose(s, 1.40636669, rtol=1e-6), (
        f"Expected s=1.40636669, got {s}"
    )

def test_theis_drawdown_positive():
    """Drawdown from pumping must always be positive."""
    s = theis_drawdown(Q=4088, T=1000, S=3e-4, r=500, t=5)
    assert s > 0, f"Drawdown should be positive, got {s}"

def test_theis_increases_with_time():
    """Drawdown should increase as pumping continues."""
    s_early = theis_drawdown(Q=4088, T=1000, S=3e-4, r=1000, t=1)
    s_late = theis_drawdown(Q=4088, T=1000, S=3e-4, r=1000, t=100)
    assert s_late > s_early, (
        f"Drawdown at t=100 ({s_late}) should exceed t=1 ({s_early})"
    )

def test_theis_decreases_with_distance():
    """Drawdown should decrease farther from the well."""
    s_near = theis_drawdown(Q=4088, T=1000, S=3e-4, r=100, t=10)
    s_far = theis_drawdown(Q=4088, T=1000, S=3e-4, r=5000, t=10)
    assert s_near > s_far, (
        f"Drawdown at r=100 ({s_near}) should exceed r=5000 ({s_far})"
    )

# Run the tests
import traceback
tests = [test_theis_known_value, test_theis_drawdown_positive,
         test_theis_increases_with_time, test_theis_decreases_with_distance]
for test in tests:
    try:
        test()
        print(f"  PASSED: {test.__name__}")
    except AssertionError as e:
        print(f"  FAILED: {test.__name__} — {e}")

print(f"\n{len(tests)} tests run.")

### Writing pytest functions for Darcy's Law

In [ ]:
def test_darcy_known_value():
    """Test Darcy flow against a hand calculation."""
    Q = darcy_flow(K=10, A=50, dh=5, dL=100)
    assert np.isclose(Q, 25.0), f"Expected Q=25.0, got {Q}"

def test_darcy_zero_gradient():
    """No head difference means no flow."""
    Q = darcy_flow(K=10, A=50, dh=0, dL=100)
    assert Q == 0.0, f"Expected Q=0 with no gradient, got {Q}"

def test_darcy_proportional_to_K():
    """Doubling K should double the flow rate."""
    Q1 = darcy_flow(K=10, A=50, dh=5, dL=100)
    Q2 = darcy_flow(K=20, A=50, dh=5, dL=100)
    assert np.isclose(Q2, 2 * Q1), (
        f"Q at K=20 ({Q2}) should be 2x Q at K=10 ({Q1})"
    )

def test_darcy_proportional_to_area():
    """Doubling cross-sectional area should double the flow rate."""
    Q1 = darcy_flow(K=10, A=50, dh=5, dL=100)
    Q2 = darcy_flow(K=10, A=100, dh=5, dL=100)
    assert np.isclose(Q2, 2 * Q1), (
        f"Q at A=100 ({Q2}) should be 2x Q at A=50 ({Q1})"
    )

# Run the tests
tests = [test_darcy_known_value, test_darcy_zero_gradient,
         test_darcy_proportional_to_K, test_darcy_proportional_to_area]
for test in tests:
    try:
        test()
        print(f"  PASSED: {test.__name__}")
    except AssertionError as e:
        print(f"  FAILED: {test.__name__} — {e}")

print(f"\n{len(tests)} tests run.")

**Key takeaway:** Unit tests let you verify AI code once and re-run the checks automatically whenever the code changes. Write tests for known values *and* for physical behaviors (monotonicity, proportionality, boundary conditions).

## 4. Strategy 3: Dimensional Analysis and Physical Plausibility

Even without a known analytical solution, you can catch many AI errors by checking whether the output is **physically plausible**.

### Dimensional analysis

Every quantity in hydrology has units. If the AI computes a flow rate, the result should have units of [L³/T]. If it computes drawdown, the result should be in [L]. A quick dimensional check can catch unit conversion errors.

| Quantity | Expected units | Typical range |
|----------|---------------|---------------|
| Drawdown ($s$) | meters | 0.01 – 100 m |
| Hydraulic conductivity ($K$) | m/day | 10⁻⁷ – 10³ m/day |
| Transmissivity ($T$) | m²/day | 10⁻⁴ – 10⁵ m²/day |
| Storativity ($S$) | dimensionless | 10⁻⁵ – 0.3 |
| Streamflow ($Q$) | m³/s or cfs | 0 – 10⁶ (varies by river) |

### Physical plausibility checks

Beyond units, certain physical constraints must hold:

- **Drawdown must be positive** (pumping lowers water levels, it does not raise them)
- **Hydraulic conductivity must be positive** (water cannot flow backward through a porous medium spontaneously)
- **Storativity must be between 0 and ~0.3** for unconfined aquifers, and much smaller (10⁻⁵ to 10⁻³) for confined aquifers
- **Flow direction follows the hydraulic gradient** (water flows from high head to low head)

In [ ]:
# Example: checking physical plausibility of AI-generated parameters
def check_theis_parameters(Q, T, S, r, t):
    """Check that Theis equation parameters are physically plausible.
    
    Returns a list of warning messages (empty if all checks pass).
    """
    warnings = []
    
    if T <= 0:
        warnings.append(f"Transmissivity T={T} must be positive")
    elif T > 1e5:
        warnings.append(f"Transmissivity T={T} m²/day is unusually high")
    
    if S <= 0 or S > 0.5:
        warnings.append(f"Storativity S={S} is outside physical range (0, 0.5]")
    
    if Q <= 0:
        warnings.append(f"Pumping rate Q={Q} should be positive for extraction")
    
    if r <= 0:
        warnings.append(f"Distance r={r} must be positive")
    
    if t <= 0:
        warnings.append(f"Time t={t} must be positive")
    
    return warnings

# Test with reasonable parameters
warnings = check_theis_parameters(Q=4088, T=1000, S=3e-4, r=1000, t=10)
print("Reasonable parameters:", "All checks passed" if not warnings else warnings)

# Test with AI-generated parameters that have issues
warnings = check_theis_parameters(Q=4088, T=-500, S=3e-4, r=1000, t=10)
print("Bad T:", warnings)

warnings = check_theis_parameters(Q=4088, T=1000, S=1.5, r=1000, t=10)
print("Bad S:", warnings)

**Key takeaway:** Dimensional analysis and plausibility checks are your first line of defense. They do not require a known solution — just domain knowledge about what values are physically reasonable. Build these checks into your workflow whenever you use AI-generated code.

## 5. Hands-on Exercise: Write pytest Tests for Hydrology Equations

Now it is your turn. Write pytest-style test functions for the Theis equation and Darcy's Law implementations above.

### Exercise 5a: Theis Equation Tests

Write test functions that check:
1. **Known value test:** Verify $s = 1.40636669$ m at $r = 1000$ m, $t = 10$ days (with $Q = 4088$ m³/day, $T = 1000$ m²/day, $S = 3 \times 10^{-4}$)
2. **Parameter bounds test:** Verify that the function produces positive drawdown for any valid positive parameters

In [ ]:
# Exercise 5a: Write your Theis equation tests here

def test_theis_known_s_value():
    """Verify drawdown matches the known analytical solution."""
    # Your code here — use theis_drawdown() and assert with np.isclose()
    pass

def test_theis_positive_drawdown():
    """Verify drawdown is positive for valid parameters."""
    # Your code here — test with several sets of valid parameters
    pass

# Uncomment to run your tests:
# for test in [test_theis_known_s_value, test_theis_positive_drawdown]:
#     try:
#         test()
#         print(f"  PASSED: {test.__name__}")
#     except (AssertionError, Exception) as e:
#         print(f"  FAILED: {test.__name__} — {e}")

### Exercise 5b: Darcy's Law Tests

Write test functions that check:
1. **Known value test:** Verify $Q = 25.0$ m³/day for $K = 10$ m/day, $A = 50$ m², $dh = 5$ m, $dL = 100$ m
2. **Dimensional consistency test:** Verify that doubling $K$ doubles $Q$ (linearity), and that zero gradient gives zero flow

In [ ]:
# Exercise 5b: Write your Darcy's Law tests here

def test_darcy_known_Q_value():
    """Verify flow rate matches the hand calculation."""
    # Your code here — use darcy_flow() and assert with np.isclose()
    pass

def test_darcy_dimensional_consistency():
    """Verify dimensional consistency: Q is proportional to K and A."""
    # Your code here — check linearity with respect to K and A
    pass

# Uncomment to run your tests:
# for test in [test_darcy_known_Q_value, test_darcy_dimensional_consistency]:
#     try:
#         test()
#         print(f"  PASSED: {test.__name__}")
#     except (AssertionError, Exception) as e:
#         print(f"  FAILED: {test.__name__} — {e}")

## 6. Test Categories for AI-Generated Code

When AI writes scientific code, there are three categories of tests you should write. Each catches a different class of error.

### Category 1: Numerical Precision Tests

AI-generated code may use approximations or accumulate floating-point errors. Use `np.isclose()` with appropriate tolerances rather than exact equality.

In [ ]:
# Numerical precision test example
def test_numerical_precision():
    """Verify Theis drawdown to 6 decimal places."""
    s = theis_drawdown(Q=4088, T=1000, S=3e-4, r=1000, t=10)
    # Use rtol for relative tolerance, atol for absolute tolerance
    assert np.isclose(s, 1.40636669, rtol=1e-6), (
        f"Precision check failed: got {s}, expected 1.40636669"
    )

# Test at multiple distances to check precision across the range
for r in [100, 500, 1000, 2000, 5000]:
    s = theis_drawdown(Q=4088, T=1000, S=3e-4, r=r, t=10)
    print(f"  r = {r:5d} m  →  s = {s:.8f} m  (s > 0: {s > 0})")

### Category 2: Parameter Handling Tests

AI code often fails to validate inputs. Test what happens with edge cases and invalid parameters.

In [ ]:
# Parameter handling test examples
def test_parameter_types():
    """Verify function handles both int and float inputs."""
    s_int = theis_drawdown(Q=4088, T=1000, S=3e-4, r=1000, t=10)
    s_float = theis_drawdown(Q=4088.0, T=1000.0, S=3e-4, r=1000.0, t=10.0)
    assert np.isclose(s_int, s_float), "Int and float inputs should give same result"

def test_array_inputs():
    """Verify function works with numpy array inputs for r and t."""
    r_array = np.array([100, 500, 1000, 2000])
    s_array = theis_drawdown(Q=4088, T=1000, S=3e-4, r=r_array, t=10)
    assert len(s_array) == len(r_array), "Output array length should match input"
    assert all(s_array > 0), "All drawdown values should be positive"

# Run parameter handling tests
for test in [test_parameter_types, test_array_inputs]:
    try:
        test()
        print(f"  PASSED: {test.__name__}")
    except Exception as e:
        print(f"  FAILED: {test.__name__} — {e}")

### Category 3: Return Type Tests

AI sometimes returns unexpected data types — a list instead of an array, a string instead of a float, or a tuple when you expected a scalar.

In [ ]:
# Return type test examples
def test_return_type_scalar():
    """Verify function returns a float for scalar inputs."""
    s = theis_drawdown(Q=4088, T=1000, S=3e-4, r=1000, t=10)
    assert isinstance(s, (float, np.floating)), (
        f"Expected float, got {type(s).__name__}"
    )

def test_return_type_array():
    """Verify function returns an array for array inputs."""
    r_array = np.array([100, 500, 1000])
    s = theis_drawdown(Q=4088, T=1000, S=3e-4, r=r_array, t=10)
    assert isinstance(s, np.ndarray), (
        f"Expected numpy array, got {type(s).__name__}"
    )

for test in [test_return_type_scalar, test_return_type_array]:
    try:
        test()
        print(f"  PASSED: {test.__name__}")
    except Exception as e:
        print(f"  FAILED: {test.__name__} — {e}")

### Summary of test categories

| Category | What it catches | Example |
|----------|----------------|---------|
| **Numerical precision** | Approximation errors, wrong constants | Using 3.14 instead of `np.pi` |
| **Parameter handling** | Missing validation, wrong types | Accepting negative $K$ without error |
| **Return type** | Wrong data structure | Returning a list when an array is expected |

A good test suite for AI-generated code includes at least one test from each category.

## 7. Rapid Feedback Loop: Using Assertions While Iterating with AI

When you are actively working with an AI assistant — generating code, tweaking it, re-generating — you do not want to switch to a separate test file every time. Instead, use **inline assertions** as a rapid feedback loop.

The pattern is simple:
1. Ask AI to generate a function
2. Immediately add assertion checks in the next cell
3. If assertions fail, refine your prompt and try again
4. Once assertions pass, move the tests to a proper test file

In [ ]:
# Rapid feedback loop example
# Step 1: AI generates a function (pre-captured)
def estimate_recharge(precip_mm, et_mm, runoff_mm):
    """Estimate groundwater recharge from water balance.
    
    recharge = precipitation - evapotranspiration - runoff
    """
    recharge = precip_mm - et_mm - runoff_mm
    return recharge

# Step 2: Immediately verify with inline assertions
result = estimate_recharge(precip_mm=1000, et_mm=600, runoff_mm=200)
assert result == 200, f"Expected 200 mm, got {result}"
assert result >= 0, f"Recharge should be non-negative for these inputs, got {result}"

# Check that the function handles the zero-recharge case
result_zero = estimate_recharge(precip_mm=800, et_mm=500, runoff_mm=300)
assert result_zero == 0, f"Expected 0 mm recharge, got {result_zero}"

print("All inline assertions passed!")
print(f"Recharge estimate: {result} mm/year")
print(f"Zero-recharge case: {result_zero} mm/year")

### The iteration workflow

```
┌─────────────────────────────────────────────┐
│  1. Write prompt describing the function    │
│  2. Get AI-generated code                   │
│  3. Add inline assertions in the next cell  │
│  4. Run the cell                            │
│     ├── Assertions pass → move on           │
│     └── Assertions fail → refine prompt,    │
│         go back to step 2                   │
└─────────────────────────────────────────────┘
```

This workflow keeps you in the notebook and gives you immediate feedback. The assertions serve as a lightweight test suite that you build up as you iterate.

**Tips for effective inline assertions:**
- Start with a known value: `assert np.isclose(result, expected_value)`
- Check the sign: `assert result > 0` (for quantities that must be positive)
- Check monotonicity: `assert f(x2) > f(x1)` when `x2 > x1` and the function should be increasing
- Check boundary conditions: `assert f(0) == known_boundary_value`

## 8. Putting It All Together

Here is a summary of the three verification strategies and when to use each:

| Strategy | When to use | Effort | Confidence |
|----------|------------|--------|------------|
| **Known parameter values** | Known equations (Theis, Darcy, Manning) | Low — look up or compute by hand | High — exact comparison |
| **Unit testing (pytest)** | Any function you plan to reuse or share | Medium — write test functions | High — automated and repeatable |
| **Dimensional analysis** | Any AI output, especially unfamiliar code | Low — quick mental check | Medium — catches gross errors |

### Recommended workflow for AI-generated hydrology code

1. **Before running AI code:** Know what the answer should look like (order of magnitude, sign, units)
2. **Immediately after:** Add inline assertions for at least one known value
3. **Before committing/sharing:** Write proper pytest tests covering known values, parameter bounds, and return types
4. **Ongoing:** Run your test suite whenever the code changes

The goal is not to distrust AI — it is to **trust but verify**. AI coding assistants are powerful tools that save time on boilerplate and syntax. But the physics is your responsibility.